In [ ]:
import os
import pandas as pd
import time
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
from scripts.train_model import circuit_training, train_five_times
from scr.qsvdd_core.data_loader import QuantumDataLoader
from scripts.test_model import test, mean_auc, best_batch, save_test_results

## Constants

In [ ]:
dataset = 'fraud'
n_train = 0 ; latent_dim = 3
# num_params_conv = 375
cost_func = 'svdd'
learning_rate = 0.001

## Dataset

In [ ]:
print("Current directory:", os.getcwd())
print("Files in this folder:", os.listdir())

In [ ]:
file_path = os.path.join("..", "data", "creditcard.csv")
df = pd.read_csv(file_path)

print("Dataset loaded successfully!\n")
print("First 5 records:\n", df.head())

In [ ]:
print(df.info())

In [ ]:
print('No Frauds', round(df['Class'].value_counts()[0]/len(df) * 100,2), '% of the dataset')
print('Frauds', round(df['Class'].value_counts()[1]/len(df) * 100,2), '% of the dataset')

In [ ]:
plt.style.use('default')

sns.countplot(x='Class', data=df)
plt.title('Class Distributions \n (0: No Fraud || 1: Fraud)', fontsize=14)

plt.show()

In [ ]:
tmp = df[['Amount','Class']].copy()
class_0 = tmp.loc[tmp['Class'] == 0]['Amount']
class_1 = tmp.loc[tmp['Class'] == 1]['Amount']

In [ ]:
class_0.describe()

In [ ]:
class_1.describe()

In [ ]:
loader = QuantumDataLoader()

X_quantum, y = loader.prepare_fraud_data(df)

print(f"Features for the circuit: {X_quantum.shape}")
print(f"Labels: {y.shape}")

In [ ]:
"""
One-class Training:
Separation into normal and fraudulent examples
QSVDD will learn what is normal.
"""
normal_indices = np.where(y == 0)[0]
abnormal_indices = np.where(y == 1)[0]

# Train
# collect 1000 normal examples for training
X_train_normal_indices = np.random.choice(normal_indices, 1000, replace=False)
X_train_normal = X_quantum[X_train_normal_indices]
y_train_normal = y[X_train_normal_indices]

# X_train contains only legitimate transactions
X_train = X_train_normal
Y_train = y_train_normal

# Test (balanced)
# The code removes 100 normal examples that were not used in training
remaining_normal_indices = list(set(normal_indices) - set(X_train_normal_indices))
X_test_normal_indices = np.random.choice(remaining_normal_indices, 100, replace=False)
X_test_normal = X_quantum[X_test_normal_indices]
y_test_normal = y[X_test_normal_indices]

# The code removes 100 fraud examples.
X_test_abnormal_indices = np.random.choice(abnormal_indices, 100, replace=False)
X_test_abnormal = X_quantum[X_test_abnormal_indices]
y_test_abnormal = y[X_test_abnormal_indices]

# creates a test set with 200 examples (50% normal, 50% fraud)
X_test = np.concatenate((X_test_normal, X_test_abnormal), axis=0)
Y_test = np.concatenate((y_test_normal, y_test_abnormal), axis=0)

center = np.zeros(latent_dim)
center_train = np.tile(center, (len(X_train), 1))

print(f'X_train shape: {X_train.shape}')
print(f'Y_train shape: {Y_train.shape}')
print(f'X_test_normal shape: {X_test_normal.shape}')
print(f'X_test_abnormal shape: {X_test_abnormal.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Y_test shape: {Y_test.shape}')
print(f'center_train shape: {center_train.shape}')

### Noiseless Training

In [ ]:
train_Xdata = X_train
train_Ydata = center_train

#### QCNN Ansatz

In [ ]:
qcnn_batch_size = 4
qcnn_steps = 16*500//qcnn_batch_size

In [ ]:
qcnn_start_time = time.time()
qcnn_loss_history, qcnn_est_params, qcnn_param_history = (
    circuit_training(X_train=train_Xdata,
                     Y_train=train_Ydata,
                     batch_size=qcnn_batch_size,
                     learning_rate=learning_rate,
                     steps=qcnn_steps,
                     ansatz='qcnn'
                     )
)
qcnn_end_time = time.time()

qcnn_total_runtime = (qcnn_end_time - qcnn_start_time) / 60
print(f"Total runtime: {qcnn_total_runtime} minutes for batch size {qcnn_batch_size}")

In [ ]:
qcnn_batch_list = [4, 8, 16, 32, 64]
qcnn_step_list = [2000, 1000, 500, 250, 125]

for i in range(len(qcnn_batch_list)):
    qcnn_batch_size = qcnn_batch_list[i]
    qcnn_steps = qcnn_step_list[i]
    qcnn_matrix_of_parameters = train_five_times(X_train=train_Xdata,
                                             Y_train=train_Ydata,
                                             batch_size=qcnn_batch_size,
                                             learning_rate=learning_rate,
                                             steps=qcnn_steps,
                                             ansatz='qcnn'
                                             )
    np.savetxt(f"../results/training/QCNN/QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}_EST_PARAMS_MEAN.npy", qcnn_matrix_of_parameters)
    print(f"--- Training BATCH {qcnn_batch_size} Completed ---")
print("--- All training batches completed ---")

#### Saving Noiseless Training with QCNN ansatz

In [ ]:
np.savetxt(f"../results/training/QCNN/QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}_LOSS_HISTORY.npy", qcnn_loss_history)
np.savetxt(f"../results/training/QCNN/QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}_EST_PARAMS.npy", qcnn_est_params)
np.savetxt(f"../results/training/QCNN/QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}_PARAM_HISTORY.npy", qcnn_param_history)

#### QAE Ansatz

In [ ]:
qae_batch_size = 4
qae_steps = 16*500//qae_batch_size

In [ ]:
qae_start_time = time.time()
qae_loss_history, qae_est_params, qae_param_history = (
    circuit_training(X_train=train_Xdata,
                     Y_train=train_Ydata,
                     batch_size=qae_batch_size,
                     learning_rate=learning_rate,
                     steps=qae_steps,
                     ansatz='qae'
                     )
)
qae_end_time = time.time()

qae_total_runtime = (qae_end_time - qae_start_time) / 60
print(f"Total runtime: {qae_total_runtime} minutes for batch size {qae_batch_size}")

In [ ]:
qae_batch_list = [4, 8, 16, 32, 64]
qae_step_list = [2000, 1000, 500, 250, 125]

for i in range(len(qae_batch_list)):
    qae_batch_size = qae_batch_list[i]
    qae_steps = qae_step_list[i]
    qae_matrix_of_parameters = train_five_times(X_train=train_Xdata,
                                             Y_train=train_Ydata,
                                             batch_size=qae_batch_size,
                                             learning_rate=learning_rate,
                                             steps=qae_steps,
                                             ansatz='qae'
                                             )
    np.savetxt(f"../results/training/QAE/QAE_B{qae_batch_size:02d}S{qae_steps}_EST_PARAMS_MEAN.npy", qae_matrix_of_parameters)
    print(f"--- Training BATCH {qae_batch_size} Completed ---")
print("--- All training batches completed ---")

#### Saving noiseless training with QAE Ansatz

In [ ]:
np.savetxt(f"../results/training/QAE/QAE_B{qae_batch_size:02d}S{qae_steps}_LOSS_HISTORY.npy", qae_loss_history)
np.savetxt(f"../results/training/QAE/QAE_B{qae_batch_size:02d}S{qae_steps}_EST_PARAMS.npy", qae_est_params)
np.savetxt(f"../results/training/QAE/QAE_B{qae_batch_size:02d}S{qae_steps}_PARAM_HISTORY.npy", qae_param_history)

In [ ]:
f_name = f"../results/training/QAE/QAE_B{qae_batch_size:02d}S{qae_steps}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="qae")

print(f'mean: {mean} | std: {std}')

#### LCQHNN Ansatz

In [ ]:
lcqhnn_batch_size = 32
lcqhnn_steps = 16*500//lcqhnn_batch_size

In [ ]:
lcqhnn_start_time = time.time()
lcqhnn_loss_history, lcqhnn_est_params, lcqhnn_param_history = (
    circuit_training(X_train=train_Xdata,
                     Y_train=train_Ydata,
                     batch_size=lcqhnn_batch_size,
                     learning_rate=learning_rate,
                     steps=lcqhnn_steps,
                     ansatz='lcqhnn'
                     )
)
lcqhnn_end_time = time.time()

lcqhnn_total_runtime = (lcqhnn_end_time - lcqhnn_start_time) / 60
print(f"Total runtime: {lcqhnn_total_runtime} minutes for batch size {lcqhnn_batch_size}")

In [ ]:
lcqhnn_batch_list = [4, 8, 16, 32, 64]
lcqhnn_step_list = [2000, 1000, 500, 250, 125]

for i in range(len(lcqhnn_batch_list)):
    lcqhnn_batch_size = lcqhnn_batch_list[i]
    lcqhnn_steps = lcqhnn_step_list[i]
    lcqhnn_matrix_of_parameters = train_five_times(X_train=train_Xdata,
                                             Y_train=train_Ydata,
                                             batch_size=lcqhnn_batch_size,
                                             learning_rate=learning_rate,
                                             steps=lcqhnn_steps,
                                             ansatz='lcqhnn'
                                             )
    np.savetxt(f"../results/training/LCQHNN/LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}_EST_PARAMS_MEAN.npy", lcqhnn_matrix_of_parameters)
    print(f"--- Training BATCH {lcqhnn_batch_size} Completed ---")
print("--- All training batches completed ---")

In [ ]:
f_name = f"../results/training/LCQHNN/LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="lcqhnn")

print(f'mean: {mean} | std: {std}')

#### saving noiseless training with QAE Ansatz

In [ ]:
np.savetxt(f"../results/training/LCQHNN/LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}_LOSS_HISTORY.npy", lcqhnn_loss_history)
np.savetxt(f"../results/training/LCQHNN/LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}_EST_PARAMS.npy", lcqhnn_est_params)
np.savetxt(f"../results/training/LCQHNN/LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}_PARAM_HISTORY.npy", lcqhnn_param_history)

### Noisy Training

#### QCNN Ansatz noisy training

In [ ]:
qcnn_start_time_noise = time.time()
qcnn_loss_history_noise, qcnn_est_params_noise, qcnn_param_history_noise = (
    circuit_training(X_train=train_Xdata,
                     Y_train=train_Ydata,
                     batch_size=qcnn_batch_size,
                     learning_rate=learning_rate,
                     steps=qcnn_steps,
                     noisy=True,
                     ansatz='qcnn'
                     )
)
qcnn_end_time_noise = time.time()

qcnn_total_runtime_noise = (qcnn_end_time_noise - qcnn_start_time_noise) / 60
print(f"Total runtime: {qcnn_total_runtime_noise} minutes for batch size {qcnn_batch_size}")

In [ ]:
qcnn_matrix_of_parameters = train_five_times(X_train=train_Xdata,
                                               Y_train=train_Ydata,
                                               batch_size=qcnn_batch_size,
                                               learning_rate=learning_rate,
                                               steps=qcnn_steps,
                                               noisy=True,
                                               ansatz='qcnn'
                                               )
np.savetxt(f"../results/training/QCNN/QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}_NOISE_EST_PARAMS_MEAN.npy",
           qcnn_matrix_of_parameters)
print("--- Training Completed ---")

#### Saving QCNN ansatz Noisy Training

In [ ]:
np.savetxt(f"../results/training/QCNN/QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}_NOISE_LOSS_HISTORY.npy", qcnn_loss_history_noise)
np.savetxt(f"../results/training/QCNN/QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}_NOISE_EST_PARAMS.npy", qcnn_est_params_noise)
np.savetxt(f"../results/training/QCNN/QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}_NOISE_PARAM_HISTORY.npy", qcnn_param_history_noise)

#### QAE Ansatz noisy training

In [ ]:
qae_start_time_noise = time.time()
qae_loss_history_noise, qae_est_params_noise, qae_param_history_noise = (
    circuit_training(X_train=train_Xdata,
                     Y_train=train_Ydata,
                     batch_size=qae_batch_size,
                     learning_rate=learning_rate,
                     steps=qae_steps,
                     noisy=True,
                     ansatz='qae'
                     )
)
qae_end_time_noise = time.time()

qae_total_runtime_noise = (qae_end_time_noise - qae_start_time_noise) / 60
print(f"Total runtime: {qae_total_runtime_noise} minutes for batch size {qae_batch_size}")

In [ ]:
qae_matrix_of_parameters = train_five_times(X_train=train_Xdata,
                                               Y_train=train_Ydata,
                                               batch_size=qae_batch_size,
                                               learning_rate=learning_rate,
                                               steps=qae_steps,
                                               noisy=True,
                                               ansatz='qae'
                                               )
np.savetxt(f"../results/training/QAE/QAE_B{qae_batch_size:02d}S{qae_steps}_NOISE_EST_PARAMS_MEAN.npy",
           qae_matrix_of_parameters)
print("--- Training Completed ---")

#### saving noisy training with QAE Ansatz

In [ ]:
np.savetxt(f"../results/training/QAE/QAE_B{qae_batch_size:02d}S{qae_steps}_NOISE_LOSS_HISTORY.npy", qae_loss_history_noise)
np.savetxt(f"../results/training/QAE/QAE_B{qae_batch_size:02d}S{qae_steps}_NOISE_EST_PARAMS.npy", qae_est_params_noise)
np.savetxt(f"../results/training/QAE/QAE_B{qae_batch_size:02d}S{qae_steps}_NOISE_PARAM_HISTORY.npy", qae_param_history_noise)

#### LCQHNN Ansatz noisy training

In [ ]:
lcqhnn_start_time_noise = time.time()
lcqhnn_loss_history_noise, lcqhnn_est_params_noise, lcqhnn_param_history_noise = (
    circuit_training(X_train=train_Xdata,
                     Y_train=train_Ydata,
                     batch_size=lcqhnn_batch_size,
                     learning_rate=learning_rate,
                     steps=lcqhnn_steps,
                     noisy=True,
                     ansatz='lcqhnn'
                     )
)
lcqhnn_end_time_noise = time.time()

lcqhnn_total_runtime_noise = (lcqhnn_end_time_noise - lcqhnn_start_time_noise) / 60
print(f"Total runtime: {lcqhnn_total_runtime_noise} minutes for batch size {lcqhnn_batch_size}")

In [ ]:
lcqhnn_matrix_of_parameters = train_five_times(X_train=train_Xdata,
                                               Y_train=train_Ydata,
                                               batch_size=lcqhnn_batch_size,
                                               learning_rate=learning_rate,
                                               steps=lcqhnn_steps,
                                               noisy=True,
                                               ansatz='lcqhnn'
                                               )
np.savetxt(f"../results/training/LCQHNN/LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}_NOISE_EST_PARAMS_MEAN.npy",
           lcqhnn_matrix_of_parameters)
print("--- Training Completed ---")

#### saving noisy training with LCQHNN Ansatz

In [ ]:
np.savetxt(f"../results/training/LCQHNN/LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}_NOISE_LOSS_HISTORY.npy", lcqhnn_loss_history_noise)
np.savetxt(f"../results/training/LCQHNN/LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}_NOISE_EST_PARAMS.npy", lcqhnn_est_params_noise)
np.savetxt(f"../results/training/LCQHNN/LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}_NOISE_PARAM_HISTORY.npy", lcqhnn_param_history_noise)

## Noiseless Test

#### QCNN Ansatz noiseless Test

#### defining best batch

In [ ]:
qcnn_test_best_batch = best_batch(method='qcnn')

In [ ]:
save_test_results(qcnn_test_best_batch, method='qcnn')

In [ ]:
best_qcnn = 'B04S2000'
f_name = f"../results/training/QCNN/QCNN_{best_qcnn}_EST_PARAMS.npy"
est_params = np.loadtxt(f_name)

auc, y_pred, y_true, fpr, tpr = test(n_train, X_test, Y_test, est_params, center_train, ansatz='qcnn')

#### QAE Ansatz noiseless Test

#### defining best batch

In [ ]:
qae_test_best_batch = best_batch(method='qae')

In [ ]:
save_test_results(qae_test_best_batch, method='qae')

In [ ]:
f_name = "../results/training/QAE/QAE_B04S2000_EST_PARAMS.npy"
qae_est_params = np.loadtxt(f_name)

auc, y_pred, y_true, fpr, tpr = test(n_train, X_test, Y_test, qae_est_params, center_train, ansatz='qae')

#### LCQHNN Ansatz noiseless Test

#### defining best batch

In [ ]:
lcqhnn_test_best_batch = best_batch(method='lcqhnn')

In [ ]:
save_test_results(lcqhnn_test_best_batch, method='lcqhnn')

## Saving Noiseless Test

In [ ]:
plt.hist(y_pred)

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d; %H:%M:%S")
print(timestamp)
plt.figure()
lw = 2
plt.plot(fpr, tpr, color='dodgerblue', lw=lw, label="{:.2f}".format(auc * 100))
plt.plot([0, 1], [0, 1], color="black", lw=lw, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()

## Noisy Test

In [ ]:
auc, y_pred, y_true, fpr, tpr = test(n_train, X_test, Y_test, qae_est_params_noise, center_train, noisy=True)

## Saving Noisy Test

In [ ]:
file_name = f"B{qae_batch_size:02d}S{qae_steps}_NOISY.txt"
results_dir = os.path.join("..", "results", "test")
file_path = os.path.join(results_dir, file_name)

os.makedirs(results_dir, exist_ok=True)

with open(file_path, "w") as f:
    f.write("Test completed.\n")̣
    f.write(f"Batch Size = {qae_batch_size} \nSteps = {qae_steps} \n")
    f.write(f"Training time: {total_runtime:.2f} minutes\n")
    f.write(f"AUC: {auc:.4f}\n")

print(f"Success! Result saved to: {file_path}")
print(f"Training time: {total_runtime:.2f} minutes")
print(f"AUC Score: {auc:.4f}")

In [ ]:
plt.hist(y_pred)

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d; %H:%M:%S")
print(timestamp)
plt.figure()
lw = 2
plt.plot(fpr, tpr, color='dodgerblue', lw=lw, label="AUC Score: {:.2f}".format(auc * 100))
plt.plot([0, 1], [0, 1], color="black", lw=lw, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve noisy")
plt.legend(loc="lower right")
plt.show()